In [1500]:
#mancano la parte di calcolo delle concentrazioni giuste
#mancano tutte le considerazioni sugli errori


In [1501]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import norm
import plotly.graph_objects as go
import pandas as pd


In [1502]:
V_BSA=148 #microL
V_PBS=2992 #microL

V_i = [1500.1, 1502.8, 1510, 1502.1, 1497, 1504.4, 1511, 1554, 1513, 1512, 1543] #prelevadop dai campioni
V_B = [1513.4, 1497.5, 1514, 1534.1, 1504, 1518, 1507.1, 1550, 1504, 1514.8, 1545.4] #preelevato dal beverone
V_f = [v_i + v_b for v_i, v_b in zip(V_i, V_B)] # vomume campione 
C_i = 2.78 * (V_BSA + V_PBS) / V_BSA  # micromolare, concentrazione iniziale BSA stock
C_f = []
σ_I_fluo=0.45 #errore percentuale spettrofluorimetrico

#errore di misura volume
σ_V = 0.1  # mL
σ_Ci = np.sqrt((σ_V**2) * (C_i**2) / ((V_BSA + V_PBS)**2))  # errore di misura della concentrazione iniziale
#errore di misura concentrazione finale
σ_Cf = []  # This will be calculated after C_f is populated



def calculate_C_f(V_i, V_f, C_i):  #concentrazione BSA campioni
    return C_i * V_i / V_f

# Calcola le concentrazioni finali 
for i in range(len(V_i)):
    if i == 0:
        C_f_value = calculate_C_f(V_i[i], V_f[i], C_i)
    else:
        C_f_value = calculate_C_f(V_i[i], V_f[i], C_f_value)
    C_f.append(C_f_value)

#errore di misura concentrazione

#errore di misura concentrazione
# Calculate σ_Cf after C_f is populated
for i in range(len(V_i)):
    σ_Cf.append(np.sqrt((σ_V**2) * (C_f[i]**2) / ((V_BSA + V_PBS)**2)))  # errore di misura della concentrazione finale

C_f_reale = C_f[2:]
print(C_f_reale, len(C_f))
#concentration = [7.5, 3.75, 1.125, 0.5, 0.25, 0.1, 0.05, 0.025, 0.00125]
print(σ_Ci)
print(C_i)

[7.343336144840021, 3.6329705629287252, 1.8122482281587142, 0.9020467954082748, 0.4516062118093845, 0.22609408928859004, 0.11338427480730419, 0.05663969324324168, 0.028297839228831084] 11
0.0018783783783783783
58.98108108108107


In [1503]:
# Lista delle colonne da leggere per ogni concentrazione
columns = [
    (0, 1), (2, 3), (4, 5), (6, 7), (8, 9), (10, 11), (12, 13), (14, 15), (16, 17), (18, 19)
]

wavelengths = []
intensities = []

# Ciclo per leggere i dati da ciascuna coppia di colonne
for col_pair in columns:
    data = pd.read_csv(
        'concentrazioni.csv',
        header=1,
        usecols=col_pair,
        names=['Lughezza donda', 'Intensità'],
        sep=',',
        decimal='.',
        skipinitialspace=True,
        nrows=291
    )
    wavelengths.append(data['Lughezza donda'])
    intensities.append(data['Intensità'])

# Assegna i dati alle variabili corrispondenti
Wavelength_75, Wavelength_375, Wavelength_1125, Wavelength_05, Wavelength_025, Wavelength_01, Wavelength_005, Wavelength_0025, Wavelength_000125, Wavelength_fondo = wavelengths
Intensity_75, Intensity_375, Intensity_1125, Intensity_05, Intensity_025, Intensity_01, Intensity_005, Intensity_0025, Intensity_000125, Intensity_fondo = intensities


In [1504]:
# import plotly.graph_objects as go

# fig = go.Figure()

# # Aggiungi il fondo al grafico
# fig.add_trace(go.Scatter(x=Wavelength_fondo, y=[fondo] * len(Wavelength_fondo), mode='lines', name='Fondo'))

# # Configura il layout del grafico
# fig.update_layout(
#     title='Rappresentazione del fondo',
#     xaxis_title='Lunghezza d\'onda (nm)',
#     yaxis_title='Intensità del fondo',
#     template='plotly_white'
# )

# # Mostra il grafico
# fig.show()

In [1505]:
# Sottrazione del fondo da ogni coppia di Wavelength e Intensity
corrected_intensities1 = []

for intensity, fondo in zip(intensities, Intensity_fondo):
    corrected_intensity1 = intensity - fondo
    corrected_intensities1.append(corrected_intensity1)

    # Crea un DataFrame Pandas con i valori corretti
    df_corrected = pd.DataFrame(corrected_intensities1).transpose()
    # Define labels if not already defined
    if 'labels' not in locals():
        labels = [f"{value:.3g}" for value in C_f_reale[:-1]]  # Adjust based on the context of C_f_reale



In [1506]:
 # Stampa la tabella
df_corrected_no_fondo = df_corrected.iloc[:, :-1]  # Rimuove l'ultima colonna che rappresenta il fondo
display(df_corrected_no_fondo)

,Intensità,Intensità,Intensità,Intensità,Intensità,Intensità,Intensità,Intensità,Intensità
0,3.500750,1.707255,2.343808,0.972800,2.160814,0.584602,0.447152,1.056590,1.329357
1,3.497508,1.679802,2.724172,1.574643,2.345600,0.705614,0.689525,0.718705,1.511142
2,3.608907,1.301299,2.112738,1.012408,2.218534,0.807022,0.610374,1.041281,1.423399
3,3.489175,1.347922,2.294178,1.000743,1.700895,0.275808,0.423534,1.011592,1.201461
4,3.264441,1.181034,2.167069,0.893306,1.376085,0.535782,0.322637,0.956263,0.647406
...,...,...,...,...,...,...,...,...,...
286,9.478392,7.352423,4.840966,2.560408,1.242702,0.606053,1.003869,0.238039,0.660963
287,9.006232,8.008359,4.457921,2.644195,1.205572,0.814538,1.127172,0.709681,0.612747
288,8.845861,7.286301,4.794618,2.339236,0.717757,0.466159,0.983141,0.506918,0.752407
289,8.598531,7.138248,4.537270,2.363506,1.180928,0.788421,1.202718,0.824096,0.596066


In [1507]:
#le code sono spaiate da 0 a 10

fig = go.Figure()

# Aggiungi le tracce per ogni concentrazione usando un ciclo
wavelengths = [Wavelength_75, Wavelength_375, Wavelength_1125, Wavelength_05, Wavelength_025, Wavelength_01, Wavelength_005, Wavelength_0025, Wavelength_000125]
corrected_intensities1 = [Intensity_75, Intensity_375, Intensity_1125, Intensity_05, Intensity_025, Intensity_01, Intensity_005, Intensity_0025, Intensity_000125]
labels = [f"{value:.3g}" for value in C_f_reale]

for wavelength,  corrected_intensities1, label in zip(wavelengths,  corrected_intensities1, labels):
    fig.add_trace(go.Scatter(x=wavelength, y= corrected_intensities1, mode='lines', name=label))

# Aggiungi titolo e etichette degli assi
fig.update_layout(
    title='Grafici di fluorescenza per diverse concentrazioni',
    xaxis_title='Lunghezza d\'onda (nm)',
    yaxis_title='Intensità (a.u.)',
    template='plotly_white'
)

# Mostra il grafico
fig.show()

In [1508]:
# Sottrazione del fondo da ogni coppia di intensità # Dovrebbe corrispondere al numero di concentrazioni
#print(corrected_intensities1)

In [1509]:
# Accoppio le code e creo un dataframe con i dati aggiornati
minimi = np.array([df_corrected_no_fondo.iloc[:, col].min() for col in range(df_corrected_no_fondo.shape[1])])
min_minimo = min(minimi)
offset = minimi - min_minimo  # Calculate the offset for each column
print("Offset:", offset)

for idx, col in enumerate(df_corrected_no_fondo.columns):
    df_corrected_no_fondo.iloc[:, idx] = df_corrected_no_fondo.iloc[:, idx] - offset[idx]

# Ensure the number of columns in df_corrected_no_fondo matches the number of labels
if df_corrected_no_fondo.shape[1] != len(labels):
    print(f"Adjusting the number of labels to match the columns in df_corrected_no_fondo.")
    labels = labels[:df_corrected_no_fondo.shape[1]]  # Adjust labels to match the number of columns

# Crea una tabella Pandas con i valori corretti
df_corrected_table = pd.DataFrame(df_corrected_no_fondo.values, columns=labels)  # Use adjusted labels
display(df_corrected_table)

fig = go.Figure()

# Aggiungi le tracce per ogni colonna del dataframe corretto
for col, label in zip(df_corrected_table.columns, labels):  # Use all labels
    fig.add_trace(go.Scatter(x=wavelengths[0], y=df_corrected_table[col], mode='lines', name=f'Concentrazione {label}'))

# Aggiungi titolo e etichette degli assi
fig.update_layout(
    title='Curve di emissione per diverse concentrazioni',
    xaxis_title='Lunghezza d\'onda (nm)',
    yaxis_title='Intensità (a.u.)',
    template='plotly_white'
)

# Mostra il grafico
fig.show()

Offset: [2.55358611e+00 8.38833168e-01 8.48759569e-01 5.59653796e-01
 4.21835527e-01 4.47418765e-02 1.89250115e-01 4.14598770e-04
 0.00000000e+00]


/var/folders/fp/skj0hq394mv28b58hfvz0rxr0000gn/T/ipykernel_70169/375874198.py:8: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,7.34,3.63,1.81,0.902,0.452,0.226,0.113,0.0566,0.0283
0,0.947164,0.868422,1.495049,0.413146,1.738978,0.539860,0.257902,1.056175,1.329357
1,0.943922,0.840969,1.875413,1.014989,1.923765,0.660872,0.500275,0.718291,1.511142
2,1.055321,0.462466,1.263978,0.452754,1.796699,0.762280,0.421124,1.040867,1.423399
3,0.935589,0.509089,1.445418,0.441089,1.279059,0.231066,0.234284,1.011178,1.201461
4,0.710855,0.342201,1.318310,0.333652,0.954250,0.491040,0.133387,0.955848,0.647406
...,...,...,...,...,...,...,...,...,...
286,6.924806,6.513590,3.992207,2.000754,0.820867,0.561311,0.814619,0.237625,0.660963
287,6.452646,7.169525,3.609162,2.084541,0.783736,0.769796,0.937922,0.709266,0.612747
288,6.292275,6.447468,3.945858,1.779583,0.295922,0.421417,0.793891,0.506504,0.752407
289,6.044945,6.299415,3.688510,1.803853,0.759092,0.743680,1.013468,0.823681,0.596066


In [1510]:
#print(df_corrected_table)

In [1511]:
# vecchio, piu o meno funziona
# making maximum point interpolation:
def max_fit_parabolic(x, λcenter, a, IMAX): # -a*(x-λcenter)**2 + IMAX
    return -a*(x-λcenter)**2 + IMAX

concentration = C_f_reale



In [1512]:
from tabulate import tabulate

# Definizione della funzione parabolica senza il parametro a
def parabola(x, b, c):
    return -x**2 + b * x + c

# Funzione per fare il fit parabolico ai 10 punti intorno al massimo
def fit_parabolic_around_max(x_data, y_data, num_points= 15):
    max_index = np.argmax(y_data)
    start_index = max(0, max_index - num_points // 2)
    end_index = min(len(x_data), max_index + num_points // 2)
    x_subset = x_data[start_index:end_index]
    y_subset = y_data[start_index:end_index]
        
    # Fit parabolico
    weights = 1 / (0.0045 * np.abs(y_subset))  # Inverse of the percentage error (45%) for weighting based on intensity values
    params, pcov = curve_fit(parabola, x_subset, y_subset, p0=[1, y_data[max_index]], sigma=weights)
    return params, pcov, x_subset, y_subset

# Lista per salvare i risultati del fit
fit_results = []
IMAX_values = []
error_IMAX_values = []

# Crea una figura Plotly
fig = go.Figure()

# Ciclo su ogni curva nel dataset
for i, col in enumerate(df_corrected_table.columns):
    x_data = wavelengths[i].values  # Use the corresponding wavelength for each column
    y_data = df_corrected_table[col].values  # Get the intensity values for the column
    params, pcov, x_subset, y_subset = fit_parabolic_around_max(x_data, y_data)
    b_fit, c_fit = params
    errors = np.sqrt(np.diag(pcov))
    error_b, error_c= errors
    
    # Calcolo della y del vertice
    y_vertex = (b_fit*b_fit + 4*c_fit)/(4)  # Poiché il coefficiente di x^2 è -1
    # Calcolo dell'errore sulla y del vertice
    error_y_vertex = np.sqrt(
        (b_fit*error_b/2)**2 +
        (error_c)**2
    )
    IMAX_values.append(y_vertex)
    error_IMAX_values.append(error_y_vertex)

    fit_results.append({'Curve': i + 1,'b': b_fit, 'c': c_fit, 'y_vertex': y_vertex})
        
    # Calcola i valori della parabola
    x_fit = np.linspace(min(x_subset), max(x_subset), 100)
    y_fit = parabola(x_fit, b_fit, c_fit)
        
    # Aggiungi la curva originale al grafico
    fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='lines', name=f'Curve {i + 1}'))
        
    # Aggiungi il fit parabolico al grafico
    fig.add_trace(go.Scatter(x=x_fit, y=y_fit, mode='lines', name=f'Fit {i + 1}'))

# Mostra il grafico
fig.update_layout(
    title='Curve con fit parabolico',
    xaxis_title='Lunghezza d\'onda (nm)',
    yaxis_title='Intensità (a.u.)',
    template='plotly_white'
)
fig.show()

# Crea un DataFrame Pandas con i risultati del fit, inclusi gli errori su IMAX
fit_results_df = pd.DataFrame(fit_results)
fit_results_df['IMAX_error'] = error_IMAX_values
display(fit_results_df)
# # Stampa gli errori sui parametri del fit
# for i, result in enumerate(fit_results):
#     print(f"Set {i + 1}:")
#     print(f"Errore su b: {error_b:.2e}")
#     print(f"Errore su c: {error_c:.2e}")
#     print(f"Errore su IMAX (y_vertex): {error_IMAX_values[i]:.2e}")
#     print("-" * 30)
# Crea una tabella in formato LaTeX con i valori di I_max e le concentrazioni

# Prepara i dati per la tabella
table_data = list(zip(concentration, IMAX_values))
headers = ["Concentrazione (µM)", "I_max"]

# Genera la tabella in formato LaTeX
latex_table = tabulate(table_data, headers=headers, tablefmt="latex")
print(latex_table)

,Curve,b,c,y_vertex,IMAX_error
0,1,939.734088,-219817.046615,957.992276,437.581962
1,2,943.224034,-221594.523390,823.371155,327.586683
2,3,942.709574,-221747.466032,427.869193,547.974417
3,4,939.572107,-220515.314914,183.621002,666.466924
4,5,937.326859,-219569.987268,75.422686,645.247858
5,6,950.948634,-226029.471755,46.354263,690.551815
6,7,955.017224,-227983.999685,30.474996,684.302111
7,8,963.018887,-231827.459596,23.884484,704.933987
8,9,972.945292,-236633.048373,22.586922,710.968280


\begin{tabular}{rr}
\hline
   Concentrazione (µM) &    I\_max \\
\hline
             7.34334   & 957.992  \\
             3.63297   & 823.371  \\
             1.81225   & 427.869  \\
             0.902047  & 183.621  \\
             0.451606  &  75.4227 \\
             0.226094  &  46.3543 \\
             0.113384  &  30.475  \\
             0.0566397 &  23.8845 \\
             0.0282978 &  22.5869 \\
\hline
\end{tabular}


In [1513]:
'''''
# Use curve fitting to find the optimal parameters λcenter, a, IMAX
params, params_covariance = curve_fit(max_fit_parabolic,Wavelength_75, Intensity_75, p0=[3, 1, 9])

# Extract the fitted parameters
λcenter_fit, a_fit, IMAX_fit = params

print("Fitted parameters:")
print(f"λcenter = {λcenter_fit}")
print(f"a = {a_fit}")
print(f"IMAX = {IMAX_fit}")

# Plot the data and the fitted curve
plt.scatter(Wavelength_75, Intensity_75, label='Data')
x_fit = np.linspace(min(Wavelength_75), max(Wavelength_75), 100)
y_fit = max_fit_parabolic(x_fit, *params)
plt.plot(Wavelength_75, Intensity_75, label='Fitted curve', color='red')
plt.legend()
plt.show()
'''''

'\'\'\n# Use curve fitting to find the optimal parameters λcenter, a, IMAX\nparams, params_covariance = curve_fit(max_fit_parabolic,Wavelength_75, Intensity_75, p0=[3, 1, 9])\n\n# Extract the fitted parameters\nλcenter_fit, a_fit, IMAX_fit = params\n\nprint("Fitted parameters:")\nprint(f"λcenter = {λcenter_fit}")\nprint(f"a = {a_fit}")\nprint(f"IMAX = {IMAX_fit}")\n\n# Plot the data and the fitted curve\nplt.scatter(Wavelength_75, Intensity_75, label=\'Data\')\nx_fit = np.linspace(min(Wavelength_75), max(Wavelength_75), 100)\ny_fit = max_fit_parabolic(x_fit, *params)\nplt.plot(Wavelength_75, Intensity_75, label=\'Fitted curve\', color=\'red\')\nplt.legend()\nplt.show()\n'

In [1514]:

# Aggiorna data_sets utilizzando i valori di df_corrected_table e le concentrazioni di C_f_reale
data_sets = [
    (wavelengths[0], df_corrected_table.iloc[:, 0]),  # Concentrazione 7.34
    (wavelengths[1], df_corrected_table.iloc[:, 1]),  # Concentrazione 3.58
    (wavelengths[2], df_corrected_table.iloc[:, 2]),  # Concentrazione 1.79
    (wavelengths[3], df_corrected_table.iloc[:, 3]),  # Concentrazione 0.89
    (wavelengths[4], df_corrected_table.iloc[:, 4]),  # Concentrazione 0.446
    (wavelengths[5], df_corrected_table.iloc[:, 5]),  # Concentrazione 0.223
    (wavelengths[6], df_corrected_table.iloc[:, 6]),  # Concentrazione 0.112
    (wavelengths[7], df_corrected_table.iloc[:, 7]),  # Concentrazione 0.0559
    (wavelengths[8], df_corrected_table.iloc[:, 8])   # Concentrazione 0.02795]
   ]   

In [1515]:


# IMAX_values = []
# # Funzione per fare il fit con i 10 valori più vicini al massimo
# def fit_closest_to_max(x_data, y_data, num_points=10):
#     max_index = np.argmax(y_data)
#     max_intensity = y_data[max_index]
#     distances = np.abs(y_data - max_intensity)
#     closest_indices = np.argsort(distances)[:num_points]
#     x_closest = x_data[closest_indices]
#     y_closest = y_data[closest_indices]
#     initial_lambda_center = x_data[max_index]
#     initial_a = 1
#     initial_IMAX = max_intensity
#     bounds = ([min(x_data), 0, 0], [max(x_data), np.inf, max_intensity * 1.1])

#     try:
#         params, _ = curve_fit(max_fit_parabolic, x_closest, y_closest, 
#                               p0=[initial_lambda_center, initial_a, initial_IMAX], bounds=bounds)
#     except RuntimeError as e:
#         print(f"Errore nel fitting per il set di dati: {e}")
#         params = [np.nan, np.nan, np.nan]
    
#     return params

# fit_results = []
# fig = go.Figure()

# # Ciclo su ogni set di dati e fai il fit
# for i, (x_data, y_data) in enumerate(data_sets):
#     params = fit_closest_to_max(x_data, y_data, num_points=10)
#     λcenter_fit, a_fit, IMAX_fit = params
#     fit_results.append({
#         'set': i+1,
#         'λcenter': λcenter_fit,
#         'a': a_fit,
#         'IMAX': IMAX_fit
#     })
#     IMAX_values.append(IMAX_fit)
    
#     max_index = np.argmax(y_data)
#     max_intensity = y_data[max_index]
#     distances = np.abs(y_data - max_intensity)
#     closest_indices = np.argsort(distances)[:15]
#     x_closest = x_data[closest_indices]
#     y_closest = y_data[closest_indices]
#     x_fit = np.linspace(min(x_closest), max(x_closest), 100)
#     y_fit = max_fit_parabolic(x_fit, *params)
    
#     # Aggiungi i dati e la curva di fit al grafico
#     fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='markers', name=f'Data set {i+1}'))
#     fig.add_trace(go.Scatter(x=x_fit, y=y_fit, mode='lines', name=f'Fitted curve {i+1}'))

# # Configura il layout del grafico
# fig.update_layout(
#     title='Fit dei dati con curve paraboliche',
#     xaxis_title='Lunghezza d\'onda (nm)',
#     yaxis_title='Intensità (a.u.)',
#     template='plotly_white'
# )

# # Mostra il grafico
# fig.show()

# # Stampa i risultati di adattamento per ogni set
# for result, label in zip(fit_results, labels):
#     print(f"Concentrazione {label} - λcenter: {result['λcenter']:.2f}, a: {result['a']:.2f}, IMAX: {result['IMAX']:.2f}")


In [1516]:
print(len(IMAX_values))
print(IMAX_values[8])
print(len(C_f_reale))
print(C_f_reale[8])


9
22.586921870475635
9
0.028297839228831084


In [1517]:

fig = go.Figure()
fig.add_trace(go.Scatter(x=C_f_reale, y=IMAX_values, mode='lines+markers', name='IMAX vs Concentration'))
fig.update_layout(
    title='IMAX vs Concentration',
    xaxis_title='Concentration',
    yaxis_title='IMAX',
    template='plotly_white'
)
fig.show()

In [1518]:
I0 = IMAX_values[8]
Cans = 4.969

def funzione_binding(P, Δη, n, KD):
    par = n * P + Cans + KD
    sqrt_term = par**2 - (4 * P * n * Cans)
    if np.any(sqrt_term < 0):
        print("Errore: La radice quadrata contiene valori negativi.")
        sqrt_term = np.maximum(sqrt_term, 0)  # Imposta un limite minimo
    sqrt = np.sqrt(sqrt_term)
    return (Δη / 2) * (par - sqrt) + I0

In [1519]:
print(IMAX_values[8])
print(C_f_reale)
print(len(error_IMAX_values))
# P = C_f_reale #ho rimosso la curva più bassa
# y = IMAX_values # ho rimosso la curva più bassa
# print("P:", P)
# print("y:", y)
# print("Lunghezza P:", len(P))
# print("Lunghezza y:", len(y))


22.586921870475635
[7.343336144840021, 3.6329705629287252, 1.8122482281587142, 0.9020467954082748, 0.4516062118093845, 0.22609408928859004, 0.11338427480730419, 0.05663969324324168, 0.028297839228831084]
9


In [1520]:
# Rimuovere l'ultimo valore da concentration e IMAX_values
P = C_f_reale[0:8]  # ho rimosso la curva più bassa
y = IMAX_values[0:8]  # ho rimosso la curva più bassa

# Esegui il fit con curve_fit
initial_params = [300, 3, 10]  # Δη = max(y), n = 3 (numero di siti di legame), KD = valore medio di P
# Increase maxfev to allow more iterations
# Increase maxfev and add bounds to constrain the parameters
popt, pcov = curve_fit(funzione_binding, P, y, p0=initial_params, sigma=error_IMAX_values[0:8], maxfev=10000)

# Parametri ottimizzati
Δη_fit_1, n_fit_1, KD_fit_1 = popt
# Calcola gli errori sui parametri del fit
errors = np.sqrt(np.diag(pcov))
error_Δη_1, error_n_1, error_KD_1 = errors



# Crea un array di valori per P per il grafico del fit
P_fit = np.linspace(min(P), max(P), 100)
y_fit = funzione_binding(P_fit, Δη_fit_1, n_fit_1, KD_fit_1)

# Crea il grafico con Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=P, y=y, mode='markers', name='Dati originali'))
fig.add_trace(go.Scatter(x=P_fit, y=y_fit, mode='lines', name='Curva di fit'))
fig.update_layout(
    title='Fit dei dati con funzione binding',
    xaxis_title='Concentraziione BSA (µM)',
    yaxis_title='IMAX',
    template='plotly_white'
)
fig.show()

# Crea un DataFrame Pandas con i risultati del fit
fit_results_df = pd.DataFrame({
    'Parameter': ['Δη', 'n', 'KD'],
    'Value': [Δη_fit_1, n_fit_1, KD_fit_1]
})
# Add the fit results to the DataFrame
fit_results_df['Error'] = [error_Δη_1, error_n_1, error_KD_1]

# Display the updated DataFrame
display(fit_results_df)
print("Matrice di covarianza (pcov):")
print(pcov)

Errore: La radice quadrata contiene valori negativi.
Errore: La radice quadrata contiene valori negativi.
Errore: La radice quadrata contiene valori negativi.
Errore: La radice quadrata contiene valori negativi.


,Parameter,Value,Error
0,Δη,186.273753,4.443261
1,n,1.123097,0.066457
2,KD,-0.034198,0.042682


Matrice di covarianza (pcov):
[[1.97425721e+01 9.44295215e-02 1.01263326e-01]
 [9.44295215e-02 4.41658147e-03 2.70777602e-03]
 [1.01263326e-01 2.70777602e-03 1.82175516e-03]]


In [1521]:
# Calcolo dell'errore percentuale di n
errore_percentuale_n = (error_n_1 / n_fit_1) * 100
print(f"Errore percentuale di n: {errore_percentuale_n:.2f}%")

Errore percentuale di n: 5.92%


In [1522]:
# for n_initial in range(i, 101):  # Ciclo sui valori iniziali di n da i a 100
#     initial_params = [max(y), n_initial, np.median(P)]  # Δη = max(y), n = n_initial, KD = valore mediano di P
#     try:
#         popt, pcov = curve_fit(funzione_binding, P, y, p0=initial_params)
#         Δη_fit, n_fit, KD_fit = popt
#         errors = np.sqrt(np.diag(pcov))
#         error_Δη, error_n, error_KD = errors
#         errore_percentuale_n = (error_n / n_fit) * 100
#         c=3
#         compatibility = abs(n_fit - c) / error_n

#         # Stampa solo i risultati in cui n_fit è compreso tra 0 e 10
#         if 0 <= n_fit <= 5 and error_n < n_fit:
#             print(f"n_initial: {n_initial}, n_fit: {n_fit:.2f}± {error_n:.2e}, {errore_percentuale_n:.2f}%, compatibilità: {compatibility:.2f}")
#     except RuntimeError as e:
#         # Ignora errori di fitting
#         pass


In [1523]:
n = 3  # Valore di riferimento
compatibility = abs(n_fit_1 - n) / error_n_1
print(f"Compatibilità tra n_fit e n=3: {compatibility:.2f}")
errore_percentuale_compatibility = (compatibility / n) * 100
print(f"Errore percentuale di compatibilità: {errore_percentuale_compatibility:.2f}%")

Compatibilità tra n_fit e n=3: 28.24
Errore percentuale di compatibilità: 941.41%


In [1524]:
KA=1/KD_fit_1
σ_KD_fit_1 = error_KD_1  # Error on KD_fit_1
σ_KA = σ_KD_fit_1 / (KD_fit_1**2)  # Error propagation formula for KA = 1 / KD
print(f"KA: {KA:.2e} ± {σ_KA:.2e}")

KA: -2.92e+01 ± 3.65e+01


In [1525]:
#riscrivere fissando n a 3 e soverapporre le curve 

I0 = IMAX_values[8]
Cans = 4.969

def funzione_binding_nfix(P, Δη, KD):
    par = 3*P+Cans+KD
    sqrt = np.sqrt(par**2 - (4 * P * 3 * Cans))
    return (Δη/2) * (par-sqrt) + I0

# Rimuovere l'ultimo valore da concentration e IMAX_values
P = C_f_reale
y = IMAX_values

# Esegui il fit con curve_fit
initial_params = [max(y), np.median(P)]  # Δη = max(y), n = 1, KD = valore mediano di P
popt, pcov = curve_fit(funzione_binding_nfix, P, y, p0=initial_params)

# Parametri ottimizzati
Δη_fit_2, KD_fit_2 = popt
# Calcola gli errori sui parametri del fit
errors = np.sqrt(np.diag(pcov))
error_Δη_2, error_KD_2 = errors

# Stampa gli errori
print(f"Δη: {Δη_fit_2:.2e} ± {error_Δη_2:.2e}")
print(f"KD: {KD_fit_2:.2e} ± {error_KD_2:.2e}")

# Crea un array di valori per P per il grafico del fit
P_fit = np.linspace(min(P), max(P), 100)
y_fit = funzione_binding_nfix(P_fit, Δη_fit_2, KD_fit_2)

# Crea il grafico con Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=P, y=y, mode='markers', name='Dati originali'))
fig.add_trace(go.Scatter(x=P_fit, y=y_fit, mode='lines', name='Curva di fit'))
fig.update_layout(
    title='Fit dei dati con funzione binding',
    xaxis_title='Concentration',
    yaxis_title='IMAX',
    template='plotly_white'
)
fig.show()

# Crea un DataFrame Pandas con i risultati del fit
fit_results_df = pd.DataFrame({
    'Parameter': ['Δη', 'KD'],
    'Value': [Δη_fit_2, KD_fit_2]
})


Δη: 3.07e+02 ± 6.57e+01
KD: 1.06e+01 ± 5.26e+00


In [1526]:
# Crea un array di valori per P per il grafico del fit
P_fit = np.linspace(min(P), max(P), 100)

# Calcola i valori della curva di fit per entrambe le funzioni
y_fit_binding = funzione_binding(P_fit, Δη_fit_1, n_fit_1, KD_fit_1)
y_fit_binding_nfix = funzione_binding_nfix(P_fit, Δη_fit_2, KD_fit_2)

# Crea il grafico con Plotly
fig = go.Figure()

# Aggiungi i dati originali
fig.add_trace(go.Scatter(x=P, y=y, mode='markers', name='Dati originali'))

# Aggiungi la curva di fit con funzione_binding
fig.add_trace(go.Scatter(x=P_fit, y=y_fit_binding, mode='lines', name='Curva di fit (funzione_binding)'))

# Aggiungi la curva di fit con funzione_binding_nfix
fig.add_trace(go.Scatter(x=P_fit, y=y_fit_binding_nfix, mode='lines', name='Curva di fit (funzione_binding_nfix)'))

# Configura il layout del grafico
fig.update_layout(
    title='Confronto tra curve di fit',
    xaxis_title='Concentration',
    yaxis_title='IMAX',
    template='plotly_white'
)

# Mostra il grafico
fig.show()

Errore: La radice quadrata contiene valori negativi.


In [1527]:
# ANS è l'oggetto di cui andiamo a seguire la fluorescenza. quando è legato alla BSA ha una resa quantica molto più alta, quindi la fluorescenza è molto più alta.
# quando è libero in soluzione la sua rea quantica è prossima allo zero e non fluoresce.
# spettrofluorimetro misuro I(lambda)
# DA CONTROLLARE eccito a 350 nm e misuro fluorescenza a 360-600 nm
# mi aspetto spettri con intenità progressivamente decrescente (all'inizio tutto ANS è legato alla BSA, diluendo diminuisce)
# è possibile per le prime curve che ci sia sovrabbondanza di proteina quindi anche con la prima diluizione l'ANS potrebbe essere tutto legato
# controllare di avere misura di ANS da solo alla stessa concentrazione
# analisi dati: interpolazione parabolica del picco per trovale l'intensità massima
# IMAX in funzione della concentrazione di proteina fare fit e estrarre costante di associazione e numero di siti di legame


## PROVA


In [1528]:
C_f_reale=[29.58,14.78,7.42,3.71,1.85,0.92,0.46,0.23,0.11,0]

# Lista delle colonne da leggere per ogni concentrazione
columns = [
    (0, 1), (2, 3), (4, 5), (6, 7), (8, 9), (10, 11), (12, 13), (14, 15), (16, 17), (18, 19)
]

wavelengths = []
intensities = []

# Ciclo per leggere i dati da ciascuna coppia di colonne
for col_pair in columns:
    data = pd.read_csv(
        'prova.csv',
        header=1,
        usecols=col_pair,
        names=['Lughezza donda', 'Intensità'],
        sep=',',
        decimal='.',
        skipinitialspace=True,
        nrows=291
    )
    wavelengths.append(data['Lughezza donda'])
    intensities.append(data['Intensità'])

# Assegna i dati alle variabili corrispondenti
Wavelength_30, Wavelength_15, Wavelength_7, Wavelength_4, Wavelength_2, Wavelength_1, Wavelength_05, Wavelength_02, Wavelength_01, Wavelength_fondo = wavelengths
Intensity_30, Intensity_15, Intensity_7, Intensity_4, Intensity_2, Intensity_1, Intensity_05, Intensity_02, Intensity_01, Intensity_fondo = intensities


In [1529]:
# Sottrazione del fondo da ogni coppia di Wavelength e Intensity
corrected_intensities = []

for intensity, fondo in zip(intensities, Intensity_fondo):
    corrected_intensity = intensity - fondo
    corrected_intensities.append(corrected_intensity)

# Ora corrected_intensities contiene le intensità corrette per ogni coppia

# Calcola la lunghezza degli array intensities e wavelengths
length_intensities = len(corrected_intensities)
length_wavelengths = len(wavelengths)

print(f"Lunghezza di intensities: {length_intensities}")
print(f"Lunghezza di wavelengths: {length_wavelengths}")

Lunghezza di intensities: 10
Lunghezza di wavelengths: 10


In [1530]:
#le code sono spaiate da 0 a 10

fig = go.Figure()

# Aggiungi le tracce per ogni concentrazione usando un ciclo
wavelengths = [Wavelength_30, Wavelength_15, Wavelength_7, Wavelength_4, Wavelength_2, Wavelength_1, Wavelength_05, Wavelength_02, Wavelength_01]
intensities = [Intensity_30, Intensity_15, Intensity_7, Intensity_4, Intensity_2, Intensity_1, Intensity_05, Intensity_02, Intensity_01]
labels = [f"{value:.3g}" for value in C_f_reale]

for wavelength,  corrected_intensities, label in zip(wavelengths,  corrected_intensities, labels):
    fig.add_trace(go.Scatter(x=wavelength, y= corrected_intensities, mode='lines', name=label))

# Aggiungi titolo e etichette degli assi
fig.update_layout(
    title='Grafici di assorbimento per diverse concentrazioni',
    xaxis_title='Lunghezza d\'onda (nm)',
    yaxis_title='Intensità (a.u.)',
    template='plotly_white'
)

# Mostra il grafico
fig.show()

In [1531]:
# making maximum point interpolation:
def max_fit_parabolic(x, λcenter, a, IMAX): # -a*(x-λcenter)**2 + IMAX
    return -a*(x-λcenter)**2 + IMAX

In [1532]:

data_sets = [
    (Wavelength_30, Intensity_30), 
    (Wavelength_15, Intensity_15),  
    (Wavelength_7, Intensity_7),
    (Wavelength_4, Intensity_4),
    (Wavelength_2, Intensity_2),
    (Wavelength_1, Intensity_1),
    (Wavelength_05, Intensity_05),
    (Wavelength_02, Intensity_02),
    (Wavelength_01, Intensity_01)
]

In [1533]:
#le code sono spagliate da 0 a 10


IMAX_values = []
# Funzione per fare il fit con i 10 valori più vicini al massimo
def fit_closest_to_max(x_data, y_data, num_points=20):
    max_index = np.argmax(y_data)
    max_intensity = y_data[max_index]
    distances = np.abs(y_data - max_intensity)
    closest_indices = np.argsort(distances)[:num_points]
    x_closest = x_data[closest_indices]
    y_closest = y_data[closest_indices]
    initial_lambda_center = x_data[max_index]
    initial_a = 1
    initial_IMAX = max_intensity
    bounds = ([min(x_data), 0, 0], [max(x_data), np.inf, max_intensity * 1.1])

    try:
        params, _ = curve_fit(max_fit_parabolic, x_closest, y_closest, 
                              p0=[initial_lambda_center, initial_a, initial_IMAX], bounds=bounds)
    except RuntimeError as e:
        print(f"Errore nel fitting per il set di dati: {e}")
        params = [np.nan, np.nan, np.nan]
    
    return params

fit_results = []
fig = go.Figure()

# Ciclo su ogni set di dati e fai il fit
for i, (x_data, y_data) in enumerate(data_sets):
    params = fit_closest_to_max(x_data, y_data, num_points=20)
    λcenter_fit, a_fit, IMAX_fit = params
    fit_results.append({
        'set': i+1,
        'λcenter': λcenter_fit,
        'a': a_fit,
        'IMAX': IMAX_fit
    })
    IMAX_values.append(IMAX_fit)
    
    max_index = np.argmax(y_data)
    max_intensity = y_data[max_index]
    distances = np.abs(y_data - max_intensity)
    closest_indices = np.argsort(distances)[:20]
    x_closest = x_data[closest_indices]
    y_closest = y_data[closest_indices]
    x_fit = np.linspace(min(x_closest), max(x_closest), 100)
    y_fit = max_fit_parabolic(x_fit, *params)
    
    # Aggiungi i dati e la curva di fit al grafico
    fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='markers', name=f'Data set {i+1}'))
    fig.add_trace(go.Scatter(x=x_fit, y=y_fit, mode='lines', name=f'Fitted curve {i+1}'))

# Configura il layout del grafico
fig.update_layout(
    title='Fit dei dati con curve paraboliche',
    xaxis_title='Lunghezza d\'onda (nm)',
    yaxis_title='Intensità (a.u.)',
    template='plotly_white'
)

# Mostra il grafico
fig.show()

# Stampa i risultati di adattamento per ogni set
for result, label in zip(fit_results, labels):
    print(f"Concentrazione {label} - λcenter: {result['λcenter']:.2f}, a: {result['a']:.2f}, IMAX: {result['IMAX']:.2f}")


Concentrazione 29.6 - λcenter: 470.71, a: 0.38, IMAX: 924.20
Concentrazione 14.8 - λcenter: 469.00, a: 0.42, IMAX: 928.66
Concentrazione 7.42 - λcenter: 470.23, a: 0.44, IMAX: 822.51
Concentrazione 3.71 - λcenter: 470.71, a: 0.27, IMAX: 693.71
Concentrazione 1.85 - λcenter: 469.96, a: 0.19, IMAX: 478.98
Concentrazione 0.92 - λcenter: 471.48, a: 0.08, IMAX: 244.34
Concentrazione 0.46 - λcenter: 472.36, a: 0.03, IMAX: 94.04
Concentrazione 0.23 - λcenter: 474.45, a: 0.01, IMAX: 38.75
Concentrazione 0.11 - λcenter: 477.94, a: 0.00, IMAX: 14.83


In [1534]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=C_f_reale, y=IMAX_values, mode='lines+markers', name='IMAX vs Concentration'))
fig.update_layout(
    title='IMAX vs Concentration',
    xaxis_title='Concentration',
    yaxis_title='IMAX',
    template='plotly_white'
)
fig.show()

In [1535]:
I0 = C_f_reale[9]
Cans = 0.000005

def funzione_binding(P, Δη, n, KD):
    par = n*P-Cans+KD
    sqrt = np.sqrt(par**2 - (4 * P * n * Cans))
    return (Δη/2) * (par-sqrt) + I0


In [1536]:
print(C_f_reale[:-1])
print(IMAX_values)

[29.58, 14.78, 7.42, 3.71, 1.85, 0.92, 0.46, 0.23, 0.11]
[np.float64(924.2030484947805), np.float64(928.6635984593589), np.float64(822.513826265595), np.float64(693.7083294299925), np.float64(478.9811237353584), np.float64(244.34156842531024), np.float64(94.03763545879282), np.float64(38.75122915409439), np.float64(14.825264612267205)]


In [1537]:
# Rimuovere l'ultimo valore da C_f_reale e IMAX_values per allineare le lunghezze
P = C_f_reale[:-1]
y = IMAX_values

# Esegui il fit con curve_fit
initial_params = [max(y), 3, np.median(P)]  # Δη = max(y), n = 1, KD = valore mediano di P
popt, pcov = curve_fit(funzione_binding, P, y, p0=initial_params)

# Parametri ottimizzati
Δη_fit, n_fit, KD_fit = popt

# Crea un array di valori per P per il grafico del fit
P_fit = np.linspace(min(P), max(P), 100)
y_fit = funzione_binding(P_fit, Δη_fit, n_fit, KD_fit)

# Crea il grafico con Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=P, y=y, mode='markers', name='Dati originali'))
fig.add_trace(go.Scatter(x=P_fit, y=y_fit, mode='lines', name='Curva di fit'))
fig.update_layout(
    title='Fit dei dati con funzione binding',
    xaxis_title='Concentration',
    yaxis_title='IMAX',
    template='plotly_white'
)
fig.show()

# Crea un DataFrame Pandas con i risultati del fit
fit_results_df = pd.DataFrame({
    'Parameter': ['Δη', 'n', 'KD'],
    'Value': [Δη_fit, n_fit, KD_fit]
})
print(fit_results_df)
# Calcola gli errori sui parametri del fit
errors = np.sqrt(np.diag(pcov))
error_Δη, error_n, error_KD = errors

# Stampa i risultati del fit con gli errori
print(f"Δη: {Δη_fit:.2e} ± {error_Δη:.2e}")
print(f"n: {n_fit:.2e} ± {error_n:.2e}")
print(f"KD: {KD_fit:.2e} ± {error_KD:.2e}")

  Parameter         Value
0        Δη  2.126617e+08
1         n  1.036828e+00
2        KD  2.579239e+00
Δη: 2.13e+08 ± 1.41e+07
n: 1.04e+00 ± 1.50e+00
KD: 2.58e+00 ± 3.45e+00
